# Onlinekommentar: complete API walkthrough

This independently developed package is not official, endorsed by, or affiliated with Onlinekommentar.
The [official API documentation](https://onlinekommentar.ch/en/apis) is authoritative.

Run the cells in order. The walkthrough covers:

1. Configuration, timeouts, rate limiting, and resource cleanup
2. Every JSON list filter, language and sort choice, pagination, detail and raw responses
3. All six OAI-PMH verbs, both formats, date/set filters, and continuation tokens
4. HTTP, validation and XML protocol errors, plus an offline custom transport

API examples below make live requests when **you run them**. They use discovered IDs, bound pagination
to two pages, and pause between calls. Do not send confidential search terms. Outputs are intentionally
cleared before committing because live responses can contain personal data.

## 1. Setup

From the repository root, run `uv sync`, then `uv run jupyter lab` and open this notebook.
Select the repository's `.venv` kernel. No `sys.path` modification or extra dependency is needed.

In [ ]:
# Run `uv sync` from the repo root and select the repo `.venv` as this notebook's kernel.
import onlinekommentar

print(f"onlinekommentar {onlinekommentar.__version__} ready")

In [ ]:
import time
from collections.abc import Callable
from datetime import date
from pathlib import Path
from tempfile import TemporaryDirectory
from xml.etree import ElementTree as ET

import httpx
from pydantic import ValidationError

from onlinekommentar import (
    DEFAULT_CONFIG,
    Commentary,
    CommentarySearchResult,
    OnlinekommentarClient,
    OnlinekommentarConfig,
    load_config,
)

API_OPTIONS = {
    "base_url": "https://onlinekommentar.ch",
    "timeout": 30.0,
    "rate_limit_delay": 0.5,
    "default_language": "en",
}
REQUEST_PAUSE = 0.5
MAX_PAGES = 2


def call_api[T](operation: Callable[[], T]) -> T | None:
    """Run a demo request, reporting errors without exposing request or payload values."""
    time.sleep(REQUEST_PAUSE)
    try:
        return operation()
    except httpx.HTTPStatusError as error:
        print(f"HTTP {error.response.status_code}; this example did not complete.")
    except httpx.RequestError as error:
        print(f"Transport failure ({type(error).__name__}); this example did not complete.")
    except ValidationError:
        print("Response validation failed; inspect the schema before using these records.")
    except ValueError:
        print("Invalid arguments or JSON; this example did not complete.")
    return None

### Configuration file and constructor options

`config.yaml` is resolved relative to the kernel's working directory, which may be `examples/`.
Use `config_path` for an explicit file. Missing files use package defaults; existing files must be
YAML mappings. Constructor values override file settings. The temporary file below demonstrates
all four runtime settings without creating a persistent configuration file.

`base_url` can point to another compatible server. `timeout` applies to all HTTP calls;
`rate_limit_delay` spaces requests on one client; `default_language` supplies an omitted JSON language.
The shared `call_api()` helper also pauses between separate example clients.

In [ ]:
print("Package defaults:", DEFAULT_CONFIG.model_dump())

with TemporaryDirectory() as directory:
    config_path = Path(directory) / "config.yaml"
    config_path.write_text(
        "onlinekommentar:\n"
        "  base_url: https://onlinekommentar.ch\n"
        "  timeout: 30\n"
        "  rate_limit_delay: 0.5\n"
        "  default_language: de\n",
        encoding="utf-8",
    )
    settings = load_config(config_path)
    with OnlinekommentarClient(config_path=config_path, timeout=10.0) as configured_client:
        print("File language:", settings.default_language)
        print("Effective timeout:", configured_client.timeout)

# Direct configuration construction uses the same validation.
settings = OnlinekommentarConfig(**API_OPTIONS)
print("Walkthrough language:", settings.default_language)

## 2. JSON API: list and inspect typed responses

`list_commentaries()` returns `CommentarySearchResult`. Languages are `en`, `de`, `fr`, `it`.
Sort choices are `title`, `-title`, `date`, `-date`; the API defaults to `-date` and page 1.
Choose values below rather than requesting every possible combination.

Every example closes its HTTP client with `with`. For manual lifecycle management use
`client = OnlinekommentarClient(...)`, then `try: ... finally: client.close()`.

In [ ]:
LANGUAGE = "en"  # en | de | fr | it
SORT = "-date"  # title | -title | date | -date

with OnlinekommentarClient(**API_OPTIONS) as client:
    results = call_api(
        lambda: client.list_commentaries(language=LANGUAGE, sort=SORT, page=1, request_timeout=60.0)
    )

if results is not None:
    print(f"Page {results.current_page}/{results.last_page}; total {results.total}")
    print("Per page:", results.per_page)
    print("Next/previous available:", bool(results.next_page_url), bool(results.previous_page_url))
    for item in results.commentaries[:3]:
        print(item.title, item.language, item.date)
        print("Authors/editors:", len(item.authors), len(item.editors))
        print("Act:", item.legislative_act.title if item.legislative_act else "Not supplied")

commentary_id = results.commentaries[0].commentary_id if results and results.commentaries else None
act_id = (
    next(
        (
            item.legislative_act.legislative_act_id
            for item in results.commentaries
            if item.legislative_act and item.legislative_act.legislative_act_id
        ),
        None,
    )
    if results is not None
    else None
)

### Search, act filter, language, sort and per-call timeout

Filters can be combined. Use an act ID discovered above; there is no documented act-list endpoint.
An empty result is valid. `search` is full-text search; `legislative_act` is an ID, not an act title.
`request_timeout` is available on both list methods and the generic `oai()` method.
Detail methods and named OAI helpers use the constructor timeout.

In [ ]:
with OnlinekommentarClient(**API_OPTIONS) as client:
    search_results = call_api(
        lambda: client.list_commentaries(
            language="de",
            search="universalversammlung",
            sort="title",
            page=1,
            request_timeout=60.0,
        )
    )
    filtered_results = None
    if act_id is not None:
        filtered_results = call_api(
            lambda: client.list_commentaries(
                language=LANGUAGE,
                search="Art.",
                legislative_act=act_id,
                sort="date",
                page=1,
                request_timeout=60.0,
            )
        )

for label, page_result in [("Search", search_results), ("Combined filters", filtered_results)]:
    if page_result is not None:
        print(label, "matches:", page_result.total)
        print([item.title for item in page_result.commentaries[:3]])

### Bounded JSON pagination

Increment `page` while retaining the same filters. This example reads at most two pages.
It reports when the cap or a failed request stops the traversal; it does not imply a complete export.

In [ ]:
collected = []
with OnlinekommentarClient(**API_OPTIONS) as client:
    for page_number in range(1, MAX_PAGES + 1):
        page_result = call_api(
            lambda: client.list_commentaries(language=LANGUAGE, sort=SORT, page=page_number)
        )
        if page_result is None:
            print("Pagination stopped after a failed request; results are incomplete.")
            break
        collected.extend(page_result.commentaries)
        if not page_result.next_page_url:
            print("Reached the final page.")
            break
    else:
        print(f"Stopped at the {MAX_PAGES}-page demo cap; more records may exist.")
print("Collected records:", len(collected))

### Detail, links, raw payloads and Pydantic models

`get_commentary()` accepts an API ID discovered from a list. There may be no result to fetch.
Links are returned as strings; this notebook does not download linked HTML, PDFs or attachments.
`Person` exposes `person_id` and `name`; `LegislativeAct` exposes `legislative_act_id` and `title`.
All models preserve unknown fields in `raw`.

In [ ]:
with OnlinekommentarClient(**API_OPTIONS) as client:
    detail = call_api(lambda: client.get_commentary(commentary_id)) if commentary_id else None
    raw_detail = (
        call_api(lambda: client.get_commentary_raw(commentary_id)) if commentary_id else None
    )
    raw_page = call_api(
        lambda: client.list_commentaries_raw(
            language=LANGUAGE,
            search="Art.",
            legislative_act=act_id,
            sort="-title",
            page=1,
            request_timeout=60.0,
        )
    )

if detail is not None:
    print("Title:", detail.title)
    print("API/HTML/PDF:", detail.link, detail.html_link, detail.pdf_link)
    print("Additional document count:", len(detail.additional_document_links))
    print("Modeled fields:", list(detail.model_dump(exclude={"raw"})))
    print("Original response keys:", list(detail.raw))
if raw_detail is not None:
    try:
        reconstructed = Commentary.from_json(raw_detail)
        print("Parsed raw detail:", reconstructed.title)
    except ValidationError:
        print("Raw detail was retrieved, but does not match the typed schema.")
if raw_page is not None:
    try:
        parsed_page = CommentarySearchResult.from_json(raw_page)
        print("Parsed raw page count:", len(parsed_page.commentaries))
    except ValidationError:
        print("Raw page was retrieved, but does not match the typed schema.")
if commentary_id is None:
    print("No commentary ID was available; detail requests were skipped.")

## 3. OAI-PMH: XML and protocol errors

OAI methods return XML strings. HTTP success alone does not mean protocol success: inspect `<error>`.
The helper below separates request failures, malformed XML, protocol errors and valid XML responses.
It returns `None` on failure so later examples can skip unavailable IDs or tokens.

In the 2026-09-27 compatibility check, `ListIdentifiers` returned HTTP 500 and `ListRecords`
continuation returned `badResumptionToken`; a diagnostic response contained an expired token.
These upstream failures may recur. Do not retry indefinitely or report a partial harvest as complete.

In [ ]:
OAI_NS = {"oai": "http://www.openarchives.org/OAI/2.0/"}


def read_oai(operation: Callable[[], str]) -> ET.Element | None:
    """Fetch a demo response and reject XML protocol errors before consuming records."""
    xml = call_api(operation)
    if xml is None:
        return None
    try:
        root = ET.fromstring(xml)
    except ET.ParseError:
        print("Malformed XML; this OAI request did not complete.")
        return None
    errors = root.findall("oai:error", OAI_NS)
    if errors:
        print("OAI protocol errors:", [error.get("code") for error in errors])
        return None
    if root.tag != "{http://www.openarchives.org/OAI/2.0/}OAI-PMH":
        print("Unexpected XML root; this is not an OAI-PMH response.")
        return None
    return root


def continuation_token(root: ET.Element | None) -> str | None:
    """Return a server-issued token, treating an empty token as the final page."""
    if root is None:
        return None
    return root.findtext(".//oai:resumptionToken", namespaces=OAI_NS) or None

### Identify, ListMetadataFormats and ListSets

Discover repository information, supported formats (`oai_dc`, `oai_openaire`) and set specifications.
`ListSets` supports continuation. An optional identifier narrows `ListMetadataFormats` to one record;
that variant is shown after discovering an identifier below.

In [ ]:
with OnlinekommentarClient(**API_OPTIONS) as client:
    repository = read_oai(client.identify)
    formats = read_oai(client.list_metadata_formats)
    sets = read_oai(client.list_sets)
    sets_token = continuation_token(sets)
    if sets_token:
        next_sets = read_oai(lambda: client.list_sets(resumption_token=sets_token))

if repository is not None:
    print("Repository:", repository.findtext(".//oai:repositoryName", namespaces=OAI_NS))
    print("Date granularity:", repository.findtext(".//oai:granularity", namespaces=OAI_NS))
if formats is not None:
    print(
        "Formats:", [element.text for element in formats.findall(".//oai:metadataPrefix", OAI_NS)]
    )
set_specs = (
    [element.text for element in sets.findall(".//oai:setSpec", OAI_NS) if element.text]
    if sets is not None
    else []
)
print("First set specifications:", set_specs[:5])
selected_set = set_specs[0] if set_specs else None

### ListIdentifiers: format, date range, set and continuation

Date filters are inclusive OAI repository datestamps, not the commentary's publication date.
The API documents `YYYY-MM-DD` values. Use `set_spec` from `ListSets`; omitting it searches all sets.
The same date/set options are available on `list_records()`.

In [ ]:
FROM_DATE = "2025-01-01"
UNTIL_DATE = date.today().isoformat()

with OnlinekommentarClient(**API_OPTIONS) as client:
    identifiers = read_oai(
        lambda: client.list_identifiers(
            metadata_prefix="oai_dc",
            from_date=FROM_DATE,
            until=UNTIL_DATE,
            set_spec=selected_set,
        )
    )
    identifiers_token = continuation_token(identifiers)
    if identifiers_token:
        next_identifiers = read_oai(
            lambda: client.list_identifiers(resumption_token=identifiers_token)
        )

oai_identifier = (
    identifiers.findtext(".//oai:header/oai:identifier", namespaces=OAI_NS)
    if identifiers is not None
    else None
)
print("An identifier was discovered:", oai_identifier is not None)

### ListRecords: Dublin Core and OpenAIRE

The initial call defaults to `oai_dc`; choose `oai_openaire` for OpenAIRE metadata.
This example shows both formats and a filtered call. It discovers an identifier from records as a
fallback if `ListIdentifiers` failed. Deleted records may have a header without metadata.

In [ ]:
with OnlinekommentarClient(**API_OPTIONS) as client:
    dc_records = read_oai(client.list_records)
    openaire_records = read_oai(lambda: client.list_records(metadata_prefix="oai_openaire"))
    filtered_records = read_oai(
        lambda: client.list_records(
            metadata_prefix="oai_dc",
            from_date=FROM_DATE,
            until=UNTIL_DATE,
            set_spec=selected_set,
        )
    )

for label, root in [
    ("Dublin Core", dc_records),
    ("OpenAIRE", openaire_records),
    ("Filtered", filtered_records),
]:
    if root is not None:
        print(label, "records:", len(root.findall(".//oai:record", OAI_NS)))
if oai_identifier is None and dc_records is not None:
    oai_identifier = dc_records.findtext(".//oai:header/oai:identifier", namespaces=OAI_NS)

### Record continuation: token only

A token is opaque: pass it back unchanged. Do not also supply `metadata_prefix`, `from_date`, `until`
or `set_spec`. The token represents the original selection. An absent/empty token means completion;
a protocol error means failure. This example follows at most one continuation, not a full harvest.

In [ ]:
records_token = continuation_token(dc_records)
if records_token:
    with OnlinekommentarClient(**API_OPTIONS) as client:
        next_records = read_oai(lambda: client.list_records(resumption_token=records_token))
    if next_records is not None:
        print("Continuation records:", len(next_records.findall(".//oai:record", OAI_NS)))
        print("More pages available:", bool(continuation_token(next_records)))
else:
    print("No continuation is available (last page or an earlier request failed).")

### GetRecord and record-specific formats

OAI identifiers are distinct from JSON commentary IDs. Use the identifier returned in an OAI header
instead of a stale documentation example. `get_record()` defaults to Dublin Core and accepts either
advertised format. Here we select OpenAIRE explicitly.

In [ ]:
if oai_identifier:
    with OnlinekommentarClient(**API_OPTIONS) as client:
        record_formats = read_oai(lambda: client.list_metadata_formats(identifier=oai_identifier))
        record = read_oai(
            lambda: client.get_record(identifier=oai_identifier, metadata_prefix="oai_openaire")
        )
    if record_formats is not None:
        print(
            "Record formats:",
            [element.text for element in record_formats.findall(".//oai:metadataPrefix", OAI_NS)],
        )
    if record is not None:
        print("Metadata present:", record.find(".//oai:metadata", OAI_NS) is not None)
else:
    print("No OAI identifier was available; record-specific requests were skipped.")

### Generic oai(): every protocol argument and per-call timeout

Use the generic method for explicit protocol calls or a per-call timeout. The mapping is:

| Python argument | Wire parameter | Applicable verbs |
| --- | --- | --- |
| `verb` | `verb` | All six verbs |
| `metadata_prefix` | `metadataPrefix` | GetRecord, ListIdentifiers, ListRecords |
| `identifier` | `identifier` | GetRecord; optionally ListMetadataFormats |
| `from_date` / `until` | `from` / `until` | ListIdentifiers, ListRecords |
| `set_spec` | `set` | ListIdentifiers, ListRecords |
| `resumption_token` | `resumptionToken` | ListSets, ListIdentifiers, ListRecords; token only |
| `request_timeout` | HTTP client setting | All generic calls; never sent as a query parameter |

Supply only arguments valid for the selected verb. Other protocol errors are returned as XML.

In [ ]:
with OnlinekommentarClient(**API_OPTIONS) as client:
    generic_records = read_oai(
        lambda: client.oai(
            verb="ListRecords",
            metadata_prefix="oai_dc",
            from_date=FROM_DATE,
            until=UNTIL_DATE,
            set_spec=selected_set,
            request_timeout=60.0,
        )
    )
    if oai_identifier:
        generic_record = read_oai(
            lambda: client.oai(
                verb="GetRecord",
                identifier=oai_identifier,
                metadata_prefix="oai_dc",
                request_timeout=60.0,
            )
        )
    generic_token = continuation_token(generic_records)
    if generic_token:
        generic_next = read_oai(
            lambda: client.oai(
                verb="ListRecords",
                resumption_token=generic_token,
                request_timeout=60.0,
            )
        )

## 4. Offline transport, validation and cleanup

`transport` accepts an HTTPX transport. `MockTransport` is useful for deterministic tests and
experiments without contacting the server. The following examples are entirely offline.
Validation errors are `ValueError` subclasses. Avoid logging raw responses, search terms or full
validation inputs; the earlier request helper reports only failure categories/status codes.

In [ ]:
def demo_transport(request: httpx.Request) -> httpx.Response:
    """Return a synthetic empty page without network access."""
    return httpx.Response(
        200,
        json={
            "data": [],
            "links": {"next": None, "prev": None},
            "meta": {"current_page": 1, "last_page": 1, "per_page": 50, "total": 0},
        },
    )


with OnlinekommentarClient(
    base_url="https://example.test",
    timeout=5.0,
    rate_limit_delay=0,
    default_language="fr",
    transport=httpx.MockTransport(demo_transport),
) as offline_client:
    empty_page = offline_client.list_commentaries()
    print("Offline total:", empty_page.total)
    try:
        offline_client.list_records(metadata_prefix="oai_dc", resumption_token="example-token")
    except ValueError:
        print("Conflicting continuation arguments were rejected before any request.")

try:
    OnlinekommentarConfig(timeout=0)
except ValidationError:
    print("A nonpositive timeout is rejected.")

try:
    Commentary.from_json({"id": "example", "title": "Example", "authors": [42]})
except ValidationError:
    print("Malformed author records are rejected instead of silently dropped.")

print("All example clients were closed by their context managers.")

## Next steps

Use the typed JSON methods for application data, raw JSON methods for unmodeled fields, and OAI for
metadata harvesting. Increase page caps only deliberately, retain rate limiting, and stop on failures.
The [README](../README.md) describes the client contract; [PLAN.md](../PLAN.md) tracks known upstream
issues. Clear notebook outputs before sharing or committing a run.